# 04 · Calibration, Physical Reconstruction & Uncertainty

**Sistem Fotonika — Digital Fringe Processing**  
**Iwan Cony Setiadi · Teknik Fisika**  
**Level:** S1 semester 6 dengan pengayaan riset · **Mode:** teori + implementasi + eksperimen numerik

> **Status data:** semua citra dan angka dalam notebook ini berasal dari **model/simulasi sintetis** dengan *ground truth*, bukan pengukuran instrumen nyata. Asumsi dan keterbatasan dibahas dalam setiap eksperimen.

**Tujuan:** mengonversi fase menjadi perpindahan dan regangan hanya melalui sensitivitas yang benar serta mempropagasikan ketidakpastian pengukuran.

## Metrologi ≠ visualisasi


**Masalah ilmiah:** peta fase berwarna tidak otomatis sama dengan pengukuran tinggi atau strain. Untuk menuliskan hasil dalam $\mu$m, mm, atau microstrain, kita wajib menetapkan geometri sensitivitas, panjang gelombang, media, skala spasial, titik referensi, dan ketidakpastian.

Kita memakai dua model yang **berbeda**: (A) interferometri refleksi normal untuk *out-of-plane height*; (B) kisi moiré *in-plane* untuk perpindahan dan strain. Model (A) tidak boleh digunakan sebagai rumus universal fringe projection profilometry (FPP), karena FPP memerlukan kalibrasi geometri projektor–kamera.

**Satuan konsisten:** $\lambda_0$ (panjang gelombang vakum) dalam $\mu$m, $n$ tanpa dimensi, fase dalam rad, $z$ dalam $\mu$m, dan koordinat spasial $x,y$ dalam mm. Dalam notebook ini setiap nilai pengukuran berasal dari simulasi.

## 1 · Sensitivitas interferometri refleksi normal


Pada pantulan balik sepanjang arah normal, pergeseran permukaan $\Delta z$ mengubah panjang lintasan geometris bolak-balik sebesar $2\Delta z$. Pada indeks bias homogen $n$, perubahan beda lintasan optik adalah $\Delta\mathrm{OPD}=2n\Delta z$ sehingga

$$\boxed{\Delta\phi=\frac{2\pi}{\lambda_0}\Delta\mathrm{OPD}=\frac{4\pi n}{\lambda_0}\Delta z},\quad S_\phi=\frac{\partial\phi}{\partial z}=\frac{4\pi n}{\lambda_0}.$$

Konvensi tanda diambil **fase naik jika permukaan bergerak ke arah positif sumbu $z$**; arah fase bisa berlawanan pada rancangan interferometer lain. Maka $\Delta z=(\lambda_0/4\pi n)\Delta\phi$. Untuk refleksi pada sudut selain normal atau konfigurasi non-collinear, bentuk sensitivitas berubah dan memerlukan vektor sensitivitas yang tepat. Satu fringe $2\pi$ = $\lambda_0/(2n)$ perubahan tinggi pada geometri ini, **bukan resolusi instrumen**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy import stats

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'figure.dpi': 105, 'axes.grid': False,
                     'image.interpolation': 'nearest'})
print('NumPy:', np.__version__)

In [ ]:
from skimage.restoration import unwrap_phase
from scipy.ndimage import gaussian_filter

lam_um=0.6328 # laser merah: panjang gelombang vakum dalam µm (nilai contoh)
n_medium=1.0
S_rad_per_um=4*np.pi*n_medium/lam_um
print(f'Sensitivitas teoritis {S_rad_per_um:.3f} rad/µm')
print(f'Perubahan tinggi untuk satu siklus 2π: {lam_um/(2*n_medium):.4f} µm')
assert np.isclose((2*np.pi)/S_rad_per_um,lam_um/(2*n_medium))

## 2 · Simulasi pengukuran kubah mikro dan referensi nol


Permukaan sintetis $h(x,y)$ diukur sebagai selisih tinggi terhadap area referensi datar. Kita simulasikan empat frame PSI, mengestimasi fase, meng-unwrapping, dan menetapkan konstanta fase menggunakan *reference annulus*. Referensi ini diambil pada tepi citra yang dibuat mendekati nol; pada eksperimen nyata, pemilihan referensi memerlukan bukti kerataan dan kestabilan.

**Urutan:** $h_{\rm true}\rightarrow\phi\rightarrow I_k\rightarrow \hat\phi_w\rightarrow\hat\phi_u\rightarrow \hat h$; setiap langkah punya noise dan kesalahan tersendiri.

In [ ]:
ny,nx=160,200
y,x=np.mgrid[:ny,:nx]
x_mm=(x-(nx-1)/2)*0.020
y_mm=(y-(ny-1)/2)*0.020
h_true_um=0.24*np.exp(-((x_mm/0.75)**2+(y_mm/0.65)**2)*2.0)
phi_true=S_rad_per_um*h_true_um
background=0.5+0.03*np.cos(2*np.pi*x/nx)
modulation=0.36*np.ones_like(h_true_um)
steps=np.arange(4)*np.pi/2
I_stack=(background[None,:,:]+modulation[None,:,:]*
         np.cos(phi_true[None,:,:]+steps[:,None,None]))
I_stack += np.random.default_rng(78).normal(0,0.012,I_stack.shape)
wrapped=np.arctan2(I_stack[3]-I_stack[1], I_stack[0]-I_stack[2])
phi_unw=unwrap_phase(wrapped)
reference=((x_mm/1.7)**2+(y_mm/1.35)**2)>0.90
ref_mean=np.mean(phi_unw[reference])
h_est_um=(phi_unw-ref_mean)/S_rad_per_um
h_ground_um=(phi_true-np.mean(phi_true[reference]))/S_rad_per_um
error=h_est_um-h_ground_um
rmse_height=np.sqrt(np.mean(error**2))
print('RMSE tinggi (relatif area referensi):',round(rmse_height,5),'µm')
assert rmse_height<0.015, 'Peta tinggi terlalu salah: cek konvensi atau unwrap.'
fig,axs=plt.subplots(1,4,figsize=(14.5,3.2),constrained_layout=True)
for a,arr,title in zip(axs,[I_stack[0],wrapped,h_est_um,error],
   ['Fringe $I_0$','Fase wrapped [rad]','Tinggi terestimasi [µm]','Error [µm]']):
    im=a.imshow(arr,cmap='gray' if title.startswith('Fringe') else 'viridis')
    a.set(title=title,xlabel='x [px]',ylabel='y [px]');plt.colorbar(im,ax=a)
plt.show()

## 3 · Kalibrasi sensitivitas dari benda standar


Dalam eksperimen, satuan fisik harus ditautkan ke **referensi tertelusur** dan pengambilan data berulang. Di sini kita uji rancangan kalibrasi sintetis pada $z_j$ yang diketahui. Satu titik sensor diamati sambil reflektor bergerak secara bertahap; fase dapat dibuka sepanjang urutan posisi jika tiap langkah fase antarposisi kurang dari $\pi$ dan tidak terjadi kegagalan noise.

Model kalibrasi: $\phi_j=S z_j+\phi_0+\varepsilon_j$. Kemiringan regresi memberikan $S$ rad/µm, dan hasil kalibrasi dibandingkan dengan $4\pi n/\lambda_0$. Ketidakpastian parameter regresi yang diperoleh di sini belum memasukkan ketidakpastian standar perpindahan maupun nonlinieritas aktuator.

In [ ]:
z_cal_um=np.linspace(-0.18,0.18,25)
true_cal_phase=S_rad_per_um*z_cal_um+0.6
wrapped_cal=np.angle(np.exp(1j*(true_cal_phase+
   np.random.default_rng(2026).normal(0,0.022,z_cal_um.shape))))
phase_cal=np.unwrap(wrapped_cal)
fit=stats.linregress(z_cal_um,phase_cal)
S_cal=fit.slope
print(f'S_teori={S_rad_per_um:.4f}, S_kalibrasi={S_cal:.4f} rad/µm')
print(f'Relative error={100*(S_cal/S_rad_per_um-1):.3f}%')
print(f'Uncertainty slope regresi (1σ; conditional)={fit.stderr:.4f} rad/µm')
fig,ax=plt.subplots(figsize=(7,3.5))
ax.plot(z_cal_um,phase_cal,'o',label='Fase simulasi (unwrapped)')
ax.plot(z_cal_um,fit.intercept+fit.slope*z_cal_um,'-',label='Regresi')
ax.set(xlabel='Perpindahan standar [µm]',ylabel='Fase [rad]',title='Kalibrasi sintetis')
ax.legend();plt.show()
assert abs(S_cal/S_rad_per_um-1)<0.02

## 4 · Propagasi ketidakpastian pengukuran


Dengan $z=\lambda_0\phi/(4\pi n)$, dan asumsi input independen serta pendekatan Taylor orde pertama, ketidakpastian standar gabungan adalah

$$u_c^2(z)=\left(\frac{\lambda_0}{4\pi n}\right)^2u^2(\phi)
+\left(\frac{\phi}{4\pi n}\right)^2u^2(\lambda_0)
+\left(\frac{\lambda_0\phi}{4\pi n^2}\right)^2u^2(n).$$

Jika terdapat kovarians, **tambahkan suku silang** $2(\partial z/\partial x_i)(\partial z/\partial x_j)\operatorname{cov}(x_i,x_j)$. Model ini adalah propagasi bersyarat, bukan sertifikat ketidakpastian pengukuran nyata. Error unwrapping $2\pi m$, thermal drift, bias sistematik, dan respons optik nonlinear tidak otomatis tertangkap oleh aproksimasi Gaussian.

In [ ]:
phi_test=1.2 # rad
u_phi=0.018 # rad
u_lambda=0.00003 # µm (contoh sintetik, bukan spesifikasi laser)
u_n=0.0002
terms=np.array([(lam_um/(4*np.pi*n_medium)*u_phi)**2,
    (phi_test/(4*np.pi*n_medium)*u_lambda)**2,
    (lam_um*phi_test/(4*np.pi*n_medium**2)*u_n)**2])
u_analytical=np.sqrt(np.sum(terms))
rng=np.random.default_rng(123)
N_mc=10000
phi_mc=rng.normal(phi_test,u_phi,N_mc)
lambda_mc=rng.normal(lam_um,u_lambda,N_mc)
n_mc=rng.normal(n_medium,u_n,N_mc)
z_mc=lambda_mc*phi_mc/(4*np.pi*n_mc)
u_mc=np.std(z_mc,ddof=1)
print(f'u_c analitis={u_analytical:.6f} µm; Monte Carlo={u_mc:.6f} µm')
assert abs(u_mc/u_analytical-1)<0.06
fig,ax=plt.subplots(figsize=(7,3.4))
ax.hist(z_mc,bins=50,density=True,alpha=.65)
ax.set(xlabel='z [µm]',ylabel='Probability density [1/µm]',
       title='Monte Carlo propagasi ketidakpastian (input independen)');plt.show()

## 5 · Moiré in-plane: fase bukan tinggi permukaan


Untuk perpindahan kisi **in-plane** berperiode $p$ sepanjang sumbu $x$, selisih fase signed antara *grating* objek dan referensi ideal adalah $\Delta\phi=-(2\pi/p)u_x$ jika pola objek direpresentasikan sebagai $\cos[2\pi(x-u_x)/p]$. Dengan konvensi lain tandanya dapat berbalik.

$$u_x=-\frac{p}{2\pi}\Delta\phi,\qquad \varepsilon_{xx}=\frac{\partial u_x}{\partial x}.$$

Panjang $p$ dan $u_x$ harus menggunakan **satuan yang sama**. Turunan dapat meningkatkan noise secara signifikan; kita bandingkan diferensiasi langsung dan setelah smoothing. Contoh berikut menggunakan perpindahan sintetis, bukan menganggap fase out-of-plane sebagai strain in-plane.

In [ ]:
ny,nx=128,192
mm_per_pixel=0.04
Y,X=np.mgrid[:ny,:nx]
u_true_mm=0.018*np.sin(2*np.pi*X/nx)*np.sin(np.pi*Y/ny)
p_mm=0.80
phase_moire=-(2*np.pi/p_mm)*u_true_mm
rng=np.random.default_rng(3)
phase_obs=phase_moire+rng.normal(0,0.008,phase_moire.shape)
u_obs_mm=-p_mm/(2*np.pi)*phase_obs
# np.gradient: axis 0 = y, axis 1 = x
strain_truth=np.gradient(u_true_mm,mm_per_pixel,axis=1)
strain_raw=np.gradient(u_obs_mm,mm_per_pixel,axis=1)
strain_smooth=np.gradient(gaussian_filter(u_obs_mm,sigma=1.4),mm_per_pixel,axis=1)
print('RMSE strain raw:',round(np.sqrt(np.mean((strain_raw-strain_truth)**2))*1e6,1),'µε')
print('RMSE strain smoothed:',round(np.sqrt(np.mean((strain_smooth-strain_truth)**2))*1e6,1),'µε')
fig,axes=plt.subplots(1,3,figsize=(12,3.2),constrained_layout=True)
for a,arr,t in zip(axes,[strain_truth,strain_raw,strain_smooth],
   ['Ground truth εxx [µε]','Raw gradient [µε]','Smoothed gradient [µε]']):
    im=a.imshow(arr*1e6,cmap='coolwarm',vmin=-15000,vmax=15000)
    a.set_title(t);plt.colorbar(im,ax=a)
plt.show()

## 6 · Protokol pelaporan metrologi dan latihan


Minimal laporan hasil kuantitatif harus menyatakan: (1) *measurand* dan definisi arah positif, (2) model sensitivitas dan validitas geometri, (3) referensi panjang/wavelength serta kalibrasi, (4) resolusi & sampling spasial, (5) metode estimasi fase, unwrap, dan deteksi invalid pixel, (6) budget ketidakpastian dengan korelasi bila relevan, (7) hasil dan interval ketidakpastian, (8) batas inferensi ketika data tidak memadai.

**Challenge:** bandingkan `np.gradient` dan Savitzky–Golay untuk estimasi strain ketika noise spasial berkorelasi; usulkan pengujian terhadap displacement standard fisik dan jelaskan parameter yang harus dicatat agar ketidakpastian tertelusur.

**Referensi:** NIST TN 1297, *Law of Propagation of Uncertainty*: https://www.nist.gov/pml/nist-technical-note-1297/nist-tn-1297-appendix-law-propagation-uncertainty. Untuk fringe projection: *Calibration of fringe projection profilometry: A comparative review*, Optics and Lasers in Engineering (2021), https://doi.org/10.1016/j.optlaseng.2021.106622.

<!-- AUDIT-RESEARCH-2026 -->

## 7 · Kovarians dan *uncertainty budget* yang dapat diaudit

Hukum GUM tidak membenarkan mengasumsikan semua input independen tanpa argumentasi. Untuk pengukuran $z=\lambda_0\phi/(4\pi n)$ dengan vektor input $\mathbf x=(\phi,\lambda_0,n)$, matriks kovarians $\boldsymbol\Sigma$, dan gradien sensitivitas $\mathbf J$, ketidakpastian orde pertama adalah

$$u_c^2(z)=\mathbf J\,\boldsymbol\Sigma\,\mathbf J^{\mathsf T},\qquad
\mathbf J=\left(\frac{\lambda_0}{4\pi n},\frac{\phi}{4\pi n},-\frac{\lambda_0\phi}{4\pi n^2}\right).$$

Analisis berikut sengaja mengasumsikan korelasi $\rho_{\phi,\lambda}=0.60$, $\rho_{\lambda,n}=-0.25$ dan ketidakpastian $u(\lambda)=0.006$ µm dan $u(n)=0.003$ yang sengaja diperbesar untuk menguji apakah model independen keliru; ini **bukan** kovarians sensor yang telah dikalibrasi. Metrik yang dipakai: kontribusi diag/cross-term, standar deviasi sampel Monte Carlo, dan perbandingan dengan kasus independen. *Uncertainty budget* di sini bersyarat pada model Gaussian linier dan **tidak memasukkan salah fringe order**, drift termal, maupun ketidakpastian referensi eksternal.

In [ ]:
mu_inputs=np.array([phi_test,lam_um,n_medium])
sd_inputs=np.array([u_phi,0.006,0.003]) # intentionally exaggerated synthetic uncertainties to expose covariance effects
corr=np.array([[1.,.60,0.],[.60,1.,-.25],[0.,-.25,1.]])
assert np.linalg.eigvalsh(corr).min()>0
Sigma=(sd_inputs[:,None]*corr)*sd_inputs[None,:]
J=np.array([lam_um/(4*np.pi*n_medium),phi_test/(4*np.pi*n_medium),-lam_um*phi_test/(4*np.pi*n_medium**2)])
u_correlated=float(np.sqrt(J@Sigma@J))
u_independent=float(np.sqrt(np.sum((J*sd_inputs)**2)))
rng_cov=np.random.default_rng(441)
mc_inputs=rng_cov.multivariate_normal(mu_inputs,Sigma,size=30000)
z_correlated=mc_inputs[:,1]*mc_inputs[:,0]/(4*np.pi*mc_inputs[:,2])
u_correlated_mc=float(z_correlated.std(ddof=1))
print(f'u_indep={u_independent:.8f}, u_corr_GUM={u_correlated:.8f}, u_corr_MC={u_correlated_mc:.8f} [µm]')
assert abs(u_correlated_mc/u_correlated-1)<0.04
assert abs(u_correlated-u_independent)>0.01*u_independent
# Contribution from each diagonal and off-diagonal covariance term (signed)
parts_diag=(J**2)*np.diag(Sigma)
parts_cross=np.array([2*J[i]*J[j]*Sigma[i,j] for i,j in [(0,1),(0,2),(1,2)]])
assert np.isclose(parts_diag.sum()+parts_cross.sum(),u_correlated**2,rtol=1e-10)
import pandas as pd
budget=pd.DataFrame({'Kontribusi':['fase','panjang gelombang','indeks bias','cov(fase,lambda)','cov(fase,n)','cov(lambda,n)'],
 'Variance term [µm²]':np.r_[parts_diag,parts_cross]})
print(budget.to_string(index=False,float_format=lambda z:f'{z:.3e}'))
fig,ax=plt.subplots(figsize=(8,3.5))
ax.barh(budget['Kontribusi'],budget['Variance term [µm²]'])
ax.set(xlabel='Suku varians signed [µm²]',title='Budget bersyarat dengan kovarians (sintetis)');plt.show()